# Outlier Detection

## Overview

An outlier deviates markedly from the bulk of the data. Detection methods differ by whether they assume a distribution and whether they operate on one variable or many.

| Method | Type | Best for |
|---|---|---|
| IQR fence | Univariate, non-param | Skewed data |
| Z-score | Univariate, param | Symmetric, approx. normal |
| Modified Z (MAD) | Univariate, robust | Data already containing outliers |
| Isolation Forest | Multivariate | High-dim, large n |
| Local Outlier Factor | Multivariate | Varying-density clusters |
| Mahalanobis distance | Multivariate, param | Correlated, approx. normal |

**Key principle:** detection is not deletion. Flagged values require investigation.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

rng = np.random.default_rng(42)
sns.set_theme(style="whitegrid")

n = 250
sites = pd.DataFrame({
    "site_id":    [f"SITE_{i:03d}" for i in range(1, n+1)],
    "nitrate":    rng.gamma(2, 2, n).round(2),
    "phosphorus": rng.gamma(1.5, 1.5, n).round(2),
    "ph":         rng.normal(7.2, 0.4, n).round(2),
    "richness":   rng.integers(5, 35, n),
})
sites.loc[[5, 42, 111], "nitrate"]   += [18, 22, 15]
sites.loc[[18], "ph"]                -= 3.5
sites.loc[[200, 201], ["nitrate","phosphorus"]] *= 6
print(sites.describe().round(2))

---
## Univariate Methods: IQR, Z-score, MAD

In [ ]:
results = pd.DataFrame({"site_id": sites["site_id"]})
for col in ["nitrate", "phosphorus", "ph"]:
    x = sites[col]
    Q1, Q3 = x.quantile(0.25), x.quantile(0.75)
    IQR = Q3 - Q1
    results[col+"_iqr"] = ((x < Q1-1.5*IQR) | (x > Q3+1.5*IQR)).astype(int)
    results[col+"_z"]   = (np.abs(stats.zscore(x)) > 3).astype(int)
    med = x.median(); mad = (x-med).abs().median()
    results[col+"_mad"] = ((0.6745*(x-med)/mad).abs() > 3.5).astype(int)
flag_cols = [c for c in results.columns if c != "site_id"]
print("Flags per method:"); print(results[flag_cols].sum())
results["total_flags"] = results[flag_cols].sum(axis=1)
print("\nSites with >= 2 flags:")
print(results[results["total_flags"] >= 2].head(8))

---
## Multivariate Detection: Isolation Forest, LOF, Mahalanobis

In [ ]:
feat = ["nitrate", "phosphorus", "ph"]
X   = sites[feat].values

ifor = IsolationForest(contamination=0.05, random_state=42)
sites["iforest"] = (ifor.fit_predict(X) == -1).astype(int)

lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05)
sites["lof_flag"]  = (lof.fit_predict(X) == -1).astype(int)
sites["lof_score"] = -lof.negative_outlier_factor_

cov_inv = np.linalg.inv(np.cov(X.T))
diff    = X - X.mean(0)
sites["mahal"]  = np.sqrt(np.einsum("ij,jk,ik->i", diff, cov_inv, diff))
chi2_t = stats.chi2.ppf(0.975, df=len(feat))
sites["mahal_flag"] = (sites["mahal"]**2 > chi2_t).astype(int)

print("Multivariate flags:")
print(sites[["iforest","lof_flag","mahal_flag"]].sum())
consensus = sites[(sites["iforest"]==1)&(sites["lof_flag"]==1)&(sites["mahal_flag"]==1)]
print(f"\nConsensus outliers: {len(consensus)}")
print(consensus[["site_id","nitrate","phosphorus","ph"]])

---
## Visualising Flagged Observations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
cols  = ["crimson" if v else "steelblue" for v in sites["iforest"]]
alpha = [0.9 if v else 0.3 for v in sites["iforest"]]
axes[0].scatter(sites["nitrate"], sites["ph"], c=cols, alpha=alpha, s=30)
for _, row in sites[sites["iforest"]==1].iterrows():
    axes[0].annotate(row["site_id"], (row["nitrate"], row["ph"]),
                     fontsize=7, color="crimson")
axes[0].set_xlabel("Nitrate"); axes[0].set_ylabel("pH")
axes[0].set_title("Isolation Forest Outliers (red)")

thresh = sites.loc[sites["lof_flag"]==1, "lof_score"].min()
axes[1].hist(sites["lof_score"], bins=40, color="steelblue", alpha=0.7)
axes[1].axvline(thresh, color="crimson", lw=1.5, ls="--",
                label=f"Threshold ({thresh:.2f})")
axes[1].set_xlabel("LOF score"); axes[1].set_ylabel("Count")
axes[1].set_title("LOF Score Distribution"); axes[1].legend()
plt.tight_layout(); plt.show()

---

## Common Pitfalls

**1. Removing flagged values without investigation**  
Statistical outlier flags indicate unusual values, not errors. Cross-check against field notes and neighbouring sites.

**2. Using Z-score on skewed data**  
Z-score uses mean and SD, which are distorted by outliers themselves. Use modified Z-score (MAD-based) instead.

**3. Hardcoding `contamination` without examining score distributions**  
`contamination=0.05` flags the top 5% regardless of whether they are genuinely anomalous. Examine the score distribution and choose a principled cutoff.

**4. Using only univariate methods**  
A site can be normal on every individual variable but anomalous in combination. Always include at least one multivariate method.

**5. Treating outlier removal as silent pre-processing**  
The presence of outliers is itself a finding. Report their count, identity, and characteristics alongside the main analysis.
---
*python_methods_library · Samantha McGarrigle · [github.com/samantha-mcgarrigle](https://github.com/samantha-mcgarrigle)*